In [11]:
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from pydantic import BaseModel, ValidationError

from model.config.core import config


def drop_na_inputs(*, input_data: pd.DataFrame) -> pd.DataFrame:
    """Check model inputs for na values and filter."""
    validated_data = input_data.copy()
    new_vars_with_na = [
        var
        for var in config.model_configuration.features
        if validated_data[var].isnull().sum() > 0
    ]
    print("aqui", new_vars_with_na)
    # validated_data.dropna(subset=new_vars_with_na, inplace=True)
    validated_data.drop(columns=new_vars_with_na, inplace=True)
    return validated_data


def validate_inputs(*, input_data: pd.DataFrame) -> Tuple[pd.DataFrame, Optional[dict]]:
    """Check model inputs for unprocessable values."""
    relevant_data = input_data[config.model_configuration.features].copy()
    print(relevant_data.head())
    validated_data = drop_na_inputs(input_data=relevant_data)
    print(validated_data.head())
    errors = None

    try:
        # replace numpy nans so that pydantic can validate
        MultipleDataInputs(
            inputs=validated_data.replace({np.nan: None}).to_dict(orient="records")
        )
    except ValidationError as error:
        print("Validation error:", error)
        errors = error.json()

    return validated_data, errors



class DataInputSchema(BaseModel):
    bedrooms: Optional[int] = None  # Number of rooms
    bathrooms: Optional[float] = None  # Number of bathrooms (includes half-baths)
    sqft_living: Optional[float] = None  # Built area in square meters
    sqft_lot: Optional[float] = None  # Lot area in square meters
    floors: Optional[float] = None  # Number of floors within the property
    waterfront: Optional[int] = None  # 1 if the property has a waterfront view, 0 otherwise
    view: Optional[int] = None  # 1 if the property has a waterfront view, 0 otherwise
    condition: Optional[int] = None  # Overall condition of the property (1-5)
    grade: Optional[int] = None  # Construction quality, 1-13
    sqft_above: Optional[float] = None  # Lot area in square meters
    sqft_basement: Optional[float] = None  # Basement area in square meters
    yr_built: Optional[int] = None  # Year the property was built
    yr_renovated: Optional[int] = None  # Year of last major renovation
    zipcode: Optional[int] = None  # Zip code of the property
    lat:Optional[float] = None  # Latitude of the property
    long: Optional[float] = None  # Longitude of the property
    sqft_living15: Optional[float] = None  # Built area in square meters of the nearest 15 neighbors
    sqft_lot15: Optional[float] = None  # Lot area in square meters of the nearest 15 neighbors

class MultipleDataInputs(BaseModel):
    inputs: List[DataInputSchema]



In [12]:
import pandas as pd

data = {
    'bedrooms': 3,
    'bathrooms': 2.0,
    'sqft_living': 50000.0,
    'sqft_lot': 75000.0,
    'floors': 1.0,
    'waterfront': None,
    'view': None,
    'condition': 3,
    'grade': 7,
    'sqft_above': 50000.0,
    'sqft_basement': None,
    'yr_built': None,
    'yr_renovated': None,
    'zipcode': None,
    'lat': 47.5112,
    'long': -122.2571,
    'sqft_living15': None,
    'sqft_lot15': None,
}

df = pd.DataFrame([data])
df = df.apply(pd.to_numeric, errors='coerce')
print(df.dtypes)
df


bedrooms           int64
bathrooms        float64
sqft_living      float64
sqft_lot         float64
floors           float64
waterfront       float64
view             float64
condition          int64
grade              int64
sqft_above       float64
sqft_basement    float64
yr_built         float64
yr_renovated     float64
zipcode          float64
lat              float64
long             float64
sqft_living15    float64
sqft_lot15       float64
dtype: object


,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,3,2.0,50000.0,75000.0,1.0,NaN,NaN,3,7,50000.0,NaN,NaN,NaN,NaN,47.5112,-122.2571,NaN,NaN


In [14]:
validated_data, errors = validate_inputs(input_data=df)
validated_data

   bedrooms  bathrooms  sqft_living  sqft_lot  floors  waterfront  view  \
0         3        2.0      50000.0   75000.0     1.0         NaN   NaN   

   condition  grade  sqft_above  sqft_basement  yr_built  yr_renovated  \
0          3      7     50000.0            NaN       NaN           NaN   

   zipcode      lat      long  sqft_living15  sqft_lot15  
0      NaN  47.5112 -122.2571            NaN         NaN  
aqui ['waterfront', 'view', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'sqft_living15', 'sqft_lot15']
   bedrooms  bathrooms  sqft_living  sqft_lot  floors  condition  grade  \
0         3        2.0      50000.0   75000.0     1.0          3      7   

   sqft_above      lat      long  
0     50000.0  47.5112 -122.2571  


,bedrooms,bathrooms,sqft_living,sqft_lot,floors,condition,grade,sqft_above,lat,long
0,3,2.0,50000.0,75000.0,1.0,3,7,50000.0,47.5112,-122.2571


In [ ]:
X = validated_data
X



,bedrooms,bathrooms,sqft_living,sqft_lot,floors,condition,grade,sqft_above,lat,long
0,3,2.0,50000.0,75000.0,1.0,3,7,50000.0,47.5112,-122.2571


In [ ]:
from model.processing.data_manager import load_pipeline, get_attribute_from_config_json
pipeline_file_name = f"{config.app_config.pipeline_save_file}{"0.0.1"}.pkl"
_avaluo_pipe = load_pipeline(file_name=pipeline_file_name)

NameError: name 'pipeline_file_name' is not defined